In [14]:
import numpy as np

d = np.load("pll_dump_correct_cycle_slips_20221009_043444.npz")

data = d["data_main"]
data_freq = d["data_freq"]
fs   = d["fs"]

print(data.shape, fs)

t = np.arange(len(data)) / fs
t_freq = np.arange(len(data_freq)) / fs

(1000000,) 125000000.0


In [15]:
# ===== FIXED DECODING =====

# 1-bit ref (NOT 16-bit anymore)
ref = (data >> 63) & 0x1

ttl = (data >> 62) & 0x1
ref_pll = (data >> 61) & 0x1

# 16-bit signed sine
sin = ((data >> 45) & 0xFFFF).astype(np.int16)

nco_sync = (data >> 44) & 0x1
nco_edge = (data >> 43) & 0x1

# 16-bit period
period = (data >> 27) & 0xFFFF

coarse_count = (data >> 25) & 0x3
coarse = (data >> 24) & 0x1

phase_align = (data >> 23) & 0x1
arm_reset = (data >> 22) & 0x1
dds_reset = (data >> 21) & 0x1
dds_reset_aligned = (data >> 20) & 0x1

pll_en = (data >> 19) & 0x1
ftw_mode = (data >> 17) & 0x3

up_act = (data >> 16) & 0x1
dn_act = (data >> 15) & 0x1

q_up = (data >> 14) & 0x1
q_dn = (data >> 13) & 0x1

In [16]:
# ===== NEW FREQ DEBUG EXTRACTION =====

# FTW (46-bit unsigned)
ftw = data_freq & ((1 << 46) - 1)

FCW = 46
f_clk = 125e6

freq_from_ftw = (ftw * f_clk) / (2**FCW)

In [17]:
# # Combine as pure bits
# corr_full = (corr_hi << 17) | corr_lo

# # Now convert to signed 33-bit
# corr = corr_full.astype(np.int64)
# sign_bit = 1 << 32
# corr[corr & sign_bit != 0] -= (1 << 33)

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "browser"

fig = make_subplots(
    rows=9, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
    subplot_titles=(
      "Input vs NCO",
      "Edges",
      "Coarse Logic",
      "FTW Mode",
      "Period",
      "PFD Activity (UP/DN)",
      "PFD Pulses",
      "FTW Selected",
      "Calculated Frequency"
    )
)

# =========================
# 1. SIGNALS
# =========================
fig.add_trace(go.Scatter(x=t, y=ref*20000, name="ref_in"), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=sin, name="sin"), row=1, col=1)

# =========================
# 2. EDGE DETECTION
# =========================
fig.add_trace(go.Scatter(x=t, y=ttl*5, name="ttl_main"), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=ref_pll*4, name="ref_pll"), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=nco_sync*3, name="nco_sync"), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=nco_edge*2, name="nco_edge"), row=2, col=1)

# =========================
# 3. COARSE LOGIC
# =========================
fig.add_trace(go.Scatter(x=t, y=coarse_count*15, name="coarse_count"), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=coarse*13, name="coarse_update"), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=phase_align*11, name="phase_align"), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=arm_reset*9, name="arm_reset"), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=dds_reset*7, name="dds_reset"), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=dds_reset_aligned*5, name="dds_reset_aligned"), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=pll_en*3, name="pll_enable"), row=3, col=1)

# =========================
# 4. FTW MODE
# =========================
fig.add_trace(go.Scatter(
    x=t,
    y=ftw_mode,
    name="ftw_mode (0 free,1 est,2 pll)"
), row=4, col=1)

# =========================
# 5. PERIOD
# =========================
fig.add_trace(go.Scatter(x=t, y=period, name="period_avg"), row=5, col=1)

# =========================
# 6. ERROR
# =========================
fig.add_trace(go.Scatter(x=t_freq, y=up_act, name="up_act"), row=6, col=1)
fig.add_trace(go.Scatter(x=t_freq, y=dn_act, name="dn_act"), row=6, col=1)

# =========================
# 7. ERROR
# =========================
fig.add_trace(go.Scatter(x=t_freq, y=q_up, name="q_up"), row=7, col=1)
fig.add_trace(go.Scatter(x=t_freq, y=q_dn, name="q_dn"), row=7, col=1)

# =========================
# 8. FTW
# =========================
fig.add_trace(go.Scatter(x=t_freq, y=ftw, name="ftw_selected"), row=8, col=1)

# # =========================
# # 8. PI LOOP CORRECTION
# # =========================
# fig.add_trace(go.Scatter(x=t_freq, y=corr, name="corr"), row=8, col=1)

# =========================
# 9. CALCULATED FREQUENCY
# =========================
fig.add_trace(go.Scatter(x=t_freq, y=freq_from_ftw, name="Frequency"), row=9, col=1)

# =========================
# LAYOUT
# =========================
fig.update_layout(
    height=1400,
    title="PLL Debug Dashboard",
    hovermode="x unified"
)

fig.show()